In [1]:
"""
DIAGNOSTIC — draws YOLO label boxes on a few training images
so you can visually verify checkbox size and position are correct.
Run this BEFORE training.
"""

import os
from PIL import Image, ImageDraw, ImageFont

# ── CONFIG ──────────────────────────────────────────
IMAGE_FOLDER    = "data"
YOLO_LABELS_DIR = "clicking_mechanism/yolo_labels"
OUTPUT_FOLDER   = "diagnostics"          # visualizations saved here
NUM_SAMPLES     = 5                      # how many images to check
# ────────────────────────────────────────────────────

CLASS_COLORS = {0: "blue", 1: "green", 2: "red"}
CLASS_NAMES  = {0: "checkbox", 1: "line", 2: "box"}

def draw_labels(img_path, label_path, out_path):
    img = Image.open(img_path).convert("RGB")
    draw = ImageDraw.Draw(img)
    W, H = img.size

    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cls, cx, cy, w, h = int(parts[0]), float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])

            # Convert normalized → pixel coords
            cx_px = cx * W;  cy_px = cy * H
            w_px  = w  * W;  h_px  = h  * H
            x1 = cx_px - w_px / 2;  y1 = cy_px - h_px / 2
            x2 = cx_px + w_px / 2;  y2 = cy_px + h_px / 2

            color = CLASS_COLORS[cls]
            draw.rectangle([x1, y1, x2, y2], outline=color, width=2)
            draw.text((x1, max(0, y1 - 12)), CLASS_NAMES[cls], fill=color)

    img.save(out_path)
    print(f"Saved: {out_path}")

def run():
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)

    # Get matched pairs only
    label_stems = set(f[:-4] for f in os.listdir(YOLO_LABELS_DIR) if f.endswith(".txt"))
    image_files = [f for f in os.listdir(IMAGE_FOLDER)
                   if f.endswith(".png") and f[:-4] in label_stems]

    if not image_files:
        print("No matched image+label pairs found. Check your paths.")
        return

    samples = image_files[:NUM_SAMPLES]
    for fname in samples:
        stem       = fname[:-4]
        img_path   = os.path.join(IMAGE_FOLDER,    fname)
        label_path = os.path.join(YOLO_LABELS_DIR, stem + ".txt")
        out_path   = os.path.join(OUTPUT_FOLDER,   f"debug_{fname}")
        draw_labels(img_path, label_path, out_path)

    print(f"\n✅ Check the '{OUTPUT_FOLDER}/' folder to verify box sizes and positions.")
    print("   Blue=checkbox | Green=line | Red=box")
    print("\nIf checkbox boxes are too small/large, adjust CHECKBOX_SIZE in step1_convert_labels.py and re-run it.")

if __name__ == "__main__":
    run()

Saved: diagnostics/debug_form117.png
Saved: diagnostics/debug_form103.png
Saved: diagnostics/debug_form81.png
Saved: diagnostics/debug_form95.png
Saved: diagnostics/debug_form42.png

✅ Check the 'diagnostics/' folder to verify box sizes and positions.
   Blue=checkbox | Green=line | Red=box

If checkbox boxes are too small/large, adjust CHECKBOX_SIZE in step1_convert_labels.py and re-run it.
